# DeepRL Monopoly — self-play vs 3x ASU (the real benchmark table)

Matches the actual competition comparison point: a competitor reportedly hit
38% win rate here with pure RL. Our reference CHAMPION.pt logs show only
17.7% in this exact table (130 games), so this is the real bar.

Pure self-play RL — ASU is called strictly as a black-box opponent via
choose_action(env). No imitation loss, no reading ASU outputs as labels,
no distillation. This table is slower than fixed-only tables (3 ASU
decisions per round instead of 0-1), so expect fewer games/hour.

## 1. Mount Drive (own checkpoint folder)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/DeepRL_Monopoly_ckpt_3asu'
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 2. Clone the repo (feature/asu-teacher-distillation branch)

In [ ]:
%cd /content
!rm -rf DeepRL_Monopoly
!git clone --branch feature/asu-teacher-distillation https://github.com/EnzeCbe/monopoly-boom.git DeepRL_Monopoly
%cd DeepRL_Monopoly

## 3. Check GPU + torch

In [ ]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 4. Train vs 3x ASU

In [ ]:
import os
OUT = f"{CHECKPOINT_DIR}/vs_3asu.pt"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

# WARNING: local CPU testing showed ~230-245s/game for this table (3 ASU
# decisions per round dominates the cost, not our own network — GPU won't
# fix this). 2000 games would take ~130 hours. Start with a small batch
# (100) and see the actual per-game rate on this runtime before committing
# to more.
!PYTHONIOENCODING=utf-8 python tools/train_vs_3asu.py \
  --algo ppo --games 100 --seed 21 \
  --checkpoint-every 25 --log-every 5 \
  --out "{OUT}"

## 5. Resume after a disconnect

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/train_vs_3asu.py \
  --algo ppo --games 20000 --seed 21 --resume \
  --checkpoint-every 25 --log-every 10 \
  --out "{OUT}"

## 6. Analyze the per-game log

In [ ]:
!PYTHONIOENCODING=utf-8 python tools/analyze_run.py "{OUT.rsplit('.', 1)[0]}_games.csv" --window 25